# 🧠 RAG Advanced — 03: Query Transformation

## The Problem with Raw Queries

When a user asks a question, they don't always phrase it
the way the document was written.

Example:
- User asks: "how does the transformer handle multiple things at once?"
- Document says: "multi-head attention allows parallel processing"

These don't share keywords → BM25 fails
They're not semantically close enough → FAISS misses

## The Solution: Query Transformation

Instead of searching with the raw query,
we **transform it first** to get better retrieval.

## Techniques we'll cover:

| Technique | What it does |
|-----------|-------------|
| **Multi-Query** | Generates multiple versions of the question |
| **RAG Fusion** | Multi-query + RRF to merge results |
| **HyDE** | Generates a hypothetical answer, searches with it |

In [1]:
!pip install langchain langchain-community langchain-huggingface -q
!pip install faiss-cpu sentence-transformers -q
!pip install langchain-groq langchain-classic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from kaggle_secrets import UserSecretsClient

# API Key
user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

# LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.3,
    max_tokens=2048
)

print("✅ Setup done!")

/tmp/ipykernel_58/3446659021.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ Setup done!


In [3]:
# Attention is all you need paper
!wget "https://arxiv.org/pdf/1706.03762" -O attention_is_all_you_need.pdf

# BERT paper 
!wget "https://arxiv.org/pdf/1810.04805" -O bert.pdf

# GPT-3 paper
!wget "https://arxiv.org/pdf/2005.14165" -O gpt3.pdf

# RAG paper itself!
!wget "https://arxiv.org/pdf/2005.11401" -O rag_paper.pdf

--2026-06-06 18:46:09--  https://arxiv.org/pdf/1706.03762
Resolving arxiv.org (arxiv.org)... 151.101.195.42, 151.101.131.42, 151.101.67.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.195.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2215244 (2.1M) [application/pdf]
Saving to: ‘attention_is_all_you_need.pdf’

attention_is_all_yo 100%[===================>]   2.11M  --.-KB/s    in 0.02s   

2026-06-06 18:46:09 (102 MB/s) - ‘attention_is_all_you_need.pdf’ saved [2215244/2215244]

--2026-06-06 18:46:09--  https://arxiv.org/pdf/1810.04805
Resolving arxiv.org (arxiv.org)... 151.101.131.42, 151.101.67.42, 151.101.195.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.131.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 775166 (757K) [application/pdf]
Saving to: ‘bert.pdf’

bert.pdf            100%[===================>] 757.00K  --.-KB/s    in 0.008s  

2026-06-06 18:46:09 (96.3 MB/s) - ‘bert.pdf’ saved [775166/775166]

--2026-06-

In [4]:
pdfs = [
    "/kaggle/working/attention_is_all_you_need.pdf",
    "/kaggle/working/bert.pdf",
    "/kaggle/working/gpt3.pdf",
    "/kaggle/working/rag_paper.pdf"
]

all_docs = []
for pdf in pdfs:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    for doc in docs:
        doc.metadata["source"] = pdf.split("/")[-1]
    all_docs.extend(docs)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)
chunks = splitter.split_documents(all_docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print(f"✅ {len(chunks)} chunks ready!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 941 chunks ready!


## Technique 1: Multi-Query

### The Problem:
One query might miss relevant chunks due to phrasing.

### The Solution:
Generate **5 different versions** of the same question.
Search with all of them → merge results → more coverage.

```
Original query: "how does attention work?"
          ↓
LLM generates:
1. "what is the attention mechanism?"
2. "how do transformers process sequences?"
3. "what is self-attention in neural networks?"
4. "how does the model focus on relevant parts?"
5. "explain multi-head attention"
          ↓
Search with all 5 → merge unique results → LLM → Answer
```

In [6]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_core.prompts import PromptTemplate

# Custom prompt to see the queries
prompt = PromptTemplate(
    input_variables=["question"],
    template="""Generate 3 different versions of this question 
    to retrieve relevant documents.
    Return only the questions, one per line.
    
    Original question: {question}"""
)

question = "how does the transformer handle multiple things at once?"

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm,
    prompt=prompt
)

# Manually generate and print queries
query_chain = prompt | llm | StrOutputParser()
generated_queries = query_chain.invoke({"question": question})

print(f"Original: {question}")
print(f"\nGenerated queries:")
print(generated_queries)
print(f"\nTotal unique chunks retrieved: {len(docs)}")

Original: how does the transformer handle multiple things at once?

Generated queries:
Here are three different versions of the question:

1. How does the Transformer architecture process multiple inputs simultaneously?
2. What mechanisms enable the Transformer model to handle multiple tasks or inputs concurrently?
3. How does the Transformer's parallel processing capabilities allow it to handle multiple elements at once?

Total unique chunks retrieved: 19


## Analysis: Multi-Query Results

Original question: "how does the transformer handle multiple things at once?"

LLM generated 3 different versions:
1. "What is the mechanism behind the transformer's ability to process multiple inputs simultaneously?"
2. "How does the transformer architecture handle parallel processing of multiple inputs or tasks?"
3. "Can you explain the transformer's capacity to handle multiple entities or sequences in a single pass?"

### Why this is powerful:
- Original query alone → ~5 chunks
- With 3 extra queries → **14 unique chunks**
- Almost 3x more coverage!

Each query uses different words but means the same thing.
This catches chunks that use different terminology.

### Key Lesson:
> Users don't always phrase questions like the document.
> Multi-Query bridges that gap by searching with multiple phrasings.

## Technique 2: HyDE
### Hypothetical Document Embeddings

### The Idea:
Instead of searching with the question —
**generate a hypothetical answer first**, then search with it.

```
Query: "how does attention work?"
          ↓
LLM generates hypothetical answer:
"Attention works by computing weights between all tokens..."
          ↓
Embed the hypothetical answer
          ↓
Search vectorstore with that embedding
          ↓
Real chunks that are similar → LLM → Real Answer
```

### Why it works:
A hypothetical answer is closer in embedding space
to the real document chunks than the question itself.

In [8]:
from langchain_core.runnables import RunnablePassthrough

# Step 1: Generate hypothetical answer
hyde_prompt = ChatPromptTemplate.from_template("""
Write a short hypothetical paragraph that would answer this question.
Write it as if it's from a research paper.

Question: {question}
Hypothetical answer:""")

# Step 2: Use hypothetical answer as search query
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

hyde_chain = (
    {"question": RunnablePassthrough()}
    | hyde_prompt
    | llm
    | StrOutputParser()
)

# Test
question = "how does attention work?"
hypothetical_answer = hyde_chain.invoke(question)

print(f"Question: {question}")
print(f"\nHypothetical Answer Generated:")
print(hypothetical_answer)

# Now search with hypothetical answer
hyde_docs = vectorstore.similarity_search(hypothetical_answer, k=5)
print(f"\nChunks found using HyDE: {len(hyde_docs)}")
print(f"\nTop chunk source: {hyde_docs[0].metadata['source']}")
print(f"Top chunk preview: {hyde_docs[0].page_content}")

Question: how does attention work?

Hypothetical Answer Generated:
Research has shown that attention operates as a dynamic filter, selectively allocating cognitive resources to relevant sensory information while suppressing irrelevant stimuli. This process is mediated by the brain's attentional network, comprising the prefrontal cortex, parietal cortex, and anterior cingulate cortex. When an individual focuses on a specific task or stimulus, the prefrontal cortex sends top-down signals to the sensory cortices, enhancing the processing of relevant information and suppressing the processing of irrelevant information. This selective amplification of sensory input is thought to occur through the modulation of neural oscillations, particularly in the gamma frequency band (30-100 Hz), which is associated with attentional control and information integration. As a result, the brain's attentional system enables individuals to efficiently process and prioritize information, facilitating learning

## Analysis: HyDE Results

### What happened?
The LLM generated a hypothetical answer about **neuroscience attention**
(brain regions, dopamine, ACC, PFC) — not transformer attention!

### Why?
The question "how does attention work?" is ambiguous.
The LLM answered from general knowledge, not document context.

### But it still worked! ✅
Despite the wrong domain in the hypothetical answer,
the top chunk came from **attention_is_all_you_need.pdf** — the right paper!

### Why did it still work?
The word "attention" in the hypothetical answer was enough
to pull the right chunks via semantic similarity.

### Fix: Be more specific in the question:

In [10]:
# More specific question
question2 = "how does attention work in transformer neural networks?"
hypothetical_answer2 = hyde_chain.invoke(question2)

print(f"Question: {question2}")
print(f"\nHypothetical Answer:")
print(hypothetical_answer2)

hyde_docs2 = vectorstore.similarity_search(hypothetical_answer2, k=5)
print(f"\nTop chunk source: {hyde_docs2[0].metadata['source']}")
print(f"Top chunk preview: {hyde_docs2[0].page_content}")

Question: how does attention work in transformer neural networks?

Hypothetical Answer:
**Attention Mechanism in Transformer Neural Networks: A Theoretical Perspective**

Our research suggests that attention in transformer neural networks operates by selectively weighting the input features based on their relevance to the current context, thereby facilitating efficient information processing and improving model performance. Specifically, the attention mechanism functions by computing a weighted sum of the input features, where the weights are determined by the dot product of the query vector and the key vector. This process allows the model to focus on the most salient features, effectively "zooming in" on the most relevant information and ignoring irrelevant details. In essence, attention in transformer networks enables the model to dynamically allocate computational resources, prioritizing the processing of critical information and thereby enhancing its ability to capture complex pat

## Analysis: Specific Question = Better HyDE

### Compare the two hypothetical answers:

| Question | Hypothetical Answer Domain | Result |
|----------|---------------------------|--------|
| "how does attention work?" | Neuroscience 🧠 | Partially correct |
| "how does attention work in transformers?" | ML/NLP ✅ | Exactly right |

### The hypothetical answer nailed it:
- "dot product of query and key vectors" ✅
- "weighted sum of input features" ✅
- "dynamically allocate computational resources" ✅

These exact concepts appear in the paper →
HyDE found the perfect chunk!

### When to use HyDE:
- ✅ Technical documents with specific terminology
- ✅ When users ask vague or short questions
- ✅ When semantic search alone misses the answer

### When NOT to use HyDE:
- ❌ Factual questions (exact numbers, dates)
- ❌ When LLM might hallucinate wrong domain
- ❌ Time-sensitive or private data questions

### Final Summary — Query Transformation:

| Technique | Best for |
|-----------|---------|
| **Multi-Query** | Vague or broad questions |
| **HyDE** | Technical concepts, research papers |
| **Both together** | Maximum coverage 🏆 |

In [11]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the context to answer. If you don't know, say so.\n\nContext:\n{context}"),
    ("human", "{input}"),
])

def get_answer(retriever, question):
    chain = create_retrieval_chain(
        retriever,
        create_stuff_documents_chain(llm, answer_prompt)
    )
    return chain.invoke({"input": question})["answer"]

question = "how does multi-head attention work in transformers?"

# Basic
basic_docs = vectorstore.similarity_search(question, k=5)
basic_context = "\n\n".join(d.page_content for d in basic_docs)
basic_answer = (answer_prompt | llm | StrOutputParser()).invoke({
    "context": basic_context, "input": question
})

# Multi-Query
mq_docs = multi_query_retriever.invoke(question)
mq_context = "\n\n".join(d.page_content for d in mq_docs)
mq_answer = (answer_prompt | llm | StrOutputParser()).invoke({
    "context": mq_context, "input": question
})

# HyDE
hyde_answer_text = hyde_chain.invoke(question)
hyde_docs = vectorstore.similarity_search(hyde_answer_text, k=5)
hyde_context = "\n\n".join(d.page_content for d in hyde_docs)
hyde_final = (answer_prompt | llm | StrOutputParser()).invoke({
    "context": hyde_context, "input": question
})

print("=" * 50)
print("🔵 Basic Retrieval:")
print(basic_answer)
print("=" * 50)
print("🟡 Multi-Query:")
print(mq_answer)
print("=" * 50)
print("🟢 HyDE:")
print(hyde_final)
print("=" * 50)

🔵 Basic Retrieval:
Multi-head attention is a key component of the Transformer architecture, and it works as follows:

1. **Splitting the Input**: The input is split into multiple attention heads, each of which processes a different aspect of the input. In the context of the Transformer, this is done by splitting the query, key, and value matrices (`Q`, `K`, and `V`) into `h` different matrices, where `h` is the number of attention heads.

2. **Attention Mechanism**: Each attention head applies the standard attention mechanism to its corresponding input. This involves computing the attention weights using the dot product of the query and key matrices, followed by a softmax function to normalize the weights.

3. **Weighted Sum**: The attention weights are used to compute a weighted sum of the value matrix for each attention head. This weighted sum is then passed through a linear transformation (i.e., a matrix multiplication) to produce the output of each attention head.

4. **Concatenati

## Final Analysis: Which Technique Wins?

### All 3 got the right answer — but quality differs:

| Technique | Answer Quality | Detail Level |
|-----------|---------------|-------------|
| 🔵 Basic | ✅ Correct | Medium — 4 steps + math |
| 🟡 Multi-Query | ✅ Correct | High — most detailed, extra context |
| 🟢 HyDE | ✅ Correct | High — clearest step-by-step |

### Key observations:

**Basic Retrieval:**
- Found the right chunks directly
- Good for well-phrased technical questions

**Multi-Query:**
- Retrieved 14 chunks vs 5
- Most comprehensive answer
- Added context about single-head vs multi-head comparison
- Best for broad or vague questions

**HyDE:**
- Clearest and most structured answer
- Step-by-step with exact math formulas
- Best for technical deep-dives

### When to use each:

| Situation | Use |
|-----------|-----|
| Simple direct question | Basic ✅ |
| Vague or broad question | Multi-Query ✅ |
| Technical concept explanation | HyDE ✅ |
| Maximum accuracy needed | Multi-Query + HyDE 🏆 |